# Gap #5: Causal Circuit Identification (Fixed – No Attention Weights)
*Ablates the output of specific attention heads to measure causal contribution.*

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
DATASET_PATH = "typosquat_dataset_full/typosquat_tool_calls.jsonl"
df_all = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df_adv = df_all[df_all['is_adversarial'] == True].sample(n=300, random_state=SEED).copy()
print(f"Sampled {len(df_adv)} adversarial entries")

texts, labels = [], []
for _, row in df_adv.iterrows():
    texts.append(row['clean_command']); labels.append(0)
    texts.append(row['typo_command']); labels.append(1)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
hidden_size = model.config.hidden_size
head_dim = hidden_size // num_heads
print(f"Layers: {num_layers}, Heads: {num_heads}, Head dim: {head_dim}")

In [ ]:
def extract_features(model, tokenizer, texts, batch_size=16):
    feats = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting baseline"):
        batch = tokenizer(texts[i:i+batch_size], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    return np.vstack(feats)

X_base = extract_features(model, tokenizer, texts)
y = np.array(labels)
probe_base = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_base, y)
auc_base = roc_auc_score(y, probe_base.predict_proba(X_base)[:, 1])
print(f"Baseline AUC: {auc_base:.4f}")

In [ ]:
layer_aucs = {}
batch_sample = texts[:64]
y_sample = y[:64]

for l in range(num_layers):
    with torch.no_grad():
        batch = tokenizer(batch_sample, padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        outs = model(**batch, output_hidden_states=True)
        feats = outs.hidden_states[l].mean(dim=1).float().cpu().numpy()
    probe = LogisticRegression(max_iter=500, random_state=SEED).fit(feats, y_sample)
    layer_aucs[l] = roc_auc_score(y_sample, probe.predict_proba(feats)[:, 1])

top_layer = max(layer_aucs, key=layer_aucs.get)
print(f"Most separable layer: L{top_layer} (AUC={layer_aucs[top_layer]:.3f})")

In [ ]:
def extract_with_head_ablation(ablated_heads):
    hooks = []
    def make_hook(layer_idx, head_idx):
        def hook_fn(module, input, output):
            bsz, seq, hidden = output.shape
            output_reshaped = output.view(bsz, seq, num_heads, head_dim)
            output_reshaped[:, :, head_idx, :] = 0.0
            return output_reshaped.view(bsz, seq, hidden)
        return hook_fn
    for layer_idx, head_idx in ablated_heads:
        layer = model.model.layers[layer_idx]
        hook = layer.self_attn.o_proj.register_forward_hook(make_hook(layer_idx, head_idx))
        hooks.append(hook)
    feats = []
    for i in tqdm(range(0, len(texts), 8), desc="Extracting with ablation"):
        batch = tokenizer(texts[i:i+8], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    for hook in hooks: hook.remove()
    return np.vstack(feats)

results = []
for h in range(num_heads):
    print(f"\nAblating L{top_layer}H{h}...")
    X_abl = extract_with_head_ablation([(top_layer, h)])
    probe_abl = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_abl, y)
    auc_abl = roc_auc_score(y, probe_abl.predict_proba(X_abl)[:, 1])
    drop = auc_base - auc_abl
    results.append({'layer': top_layer, 'head': h, 'auc': auc_abl, 'drop': drop})
    print(f"  AUC: {auc_abl:.4f} (Δ={drop:+.4f})")

res_df = pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(8,5))
bars = plt.bar(res_df['head'], res_df['drop'], color=plt.cm.viridis(res_df['drop']/res_df['drop'].max()))
plt.xlabel(f'Attention Head (Layer {top_layer})')
plt.ylabel('AUC Drop (Causal Impact)')
plt.title('Causal Contribution of Attention Heads to Typosquat Detection')
plt.axhline(y=0, color='gray', linestyle='--')
for bar, drop in zip(bars, res_df['drop']):
    if drop > 0.05:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{drop:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('causal_heads.png', dpi=150)
plt.show()

top_heads = res_df.nlargest(3, 'drop')
print("\nTop 3 causal heads:")
print(top_heads[['head', 'drop']].to_string(index=False))